# Differentiable Sorting, Ranking, Top-k, and Permutahedron (C03 Topic 13)

**Citations:** Grover et al. (2019 NeuralSort); Prillo & Eisenschlos (2020 SoftSort); Blondel et al. (2020 Permutahedron Projection); Xie et al. (2020 Differentiable Top-k); Niepert et al. (2021 I-MLE).

### Purpose & Topic Bridges
Sorting, ranking, top-$k$ selection, and quantile estimation are fundamental operations in retrieval systems, recommendation engines, and beam search decoding. However, discrete sorting operators (e.g. `argsort`) are piecewise-constant step functions whose gradients are zero almost everywhere, blocking end-to-end backpropagation. Continuous relaxations (SoftSort, NeuralSort), entropic top-$k$ bisection, soft histograms, and Permutahedron projections resolve this bottleneck by producing smooth, differentiable permutations and ranking distributions.

**Decision Boundary:** Use SoftSort / NeuralSort for end-to-end continuous rank optimization on short-to-medium sequences ($K \le 512$). For long sequences ($K > 4096$), avoid materializing $O(K^2)$ permutation matrices by deploying Permutahedron isotonic projection ($O(K \log K)$ time, $O(K)$ memory). Never set temperature $\tau \to 0$ at initialization, as extreme softmax saturation causes gradient vanishing across rank boundaries.

## 2. Mathematical Foundations & Governing Equations

1. **SoftSort Relaxation:** $P_{ij} = \text{softmax}_j(-|s_{(i)} - s_j| / \tau)$ relaxes permutation matrices, reconstructing sorted scores as $\hat{s} = P s$.
2. **NeuralSort Unimodal Relaxation:** Row $j$ defines the probability distribution over items occupying rank $j$: $P_{ji} = \text{softmax}_i(((K + 1 - 2j)s_i - \sum_k |s_i - s_k|) / \tau)$.
3. **Differentiable Top-$k$ via Bisection:** Continuous weights $w_i = \sigma((s_i - \lambda^*) / \tau)$ where root-finding enforces $\sum_{i=1}^N w_i = k$.
4. **Differentiable Soft Histogram:** $H_m = \sum_i [\sigma((x_i - (c_m - w/2))/\tau) - \sigma((x_i - (c_m + w/2))/\tau)]$, normalized to simplex $\Delta_{M-1}$.
5. **Differentiable Soft Quantile:** Median/percentile estimation via soft rank proximity: $\hat{Q}(q) = \sum_i \text{softmax}_i(-(R_i - r^*)^2 / 2\tau) s_i$.
6. **Permutahedron Projection:** $\arg\min_{y \in \mathcal{P}(\rho)} \frac{1}{2}\|y - s/\tau\|^2$ solved in $O(K \log K)$ time via Pool-Adjacent-Violators Isotonic Regression.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/13_differentiable_sorting_ranking_topk_permutahedron_and_imle.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class DifferentiableSortingUnifiedModule(nn.Module):
    def __init__(self, in_dim=4, out_dim=4):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def soft_sort(self, scores, tau=0.2):
        s_sorted, _ = torch.sort(scores, descending=True, dim=-1)
        dist = torch.abs(s_sorted.unsqueeze(-1) - scores.unsqueeze(-2))
        p_soft = torch.softmax(-dist / tau, dim=-1)
        sorted_approx = torch.matmul(p_soft, scores.unsqueeze(-1)).squeeze(-1)
        return p_soft, sorted_approx

    def neural_sort(self, scores, tau=0.5):
        B, K = scores.shape; diff = scores.unsqueeze(-1) - scores.unsqueeze(-2)
        l1_sum = torch.sum(torch.abs(diff), dim=-1)
        j_vec = torch.arange(1, K + 1, device=scores.device, dtype=scores.dtype)
        weights = (K + 1.0 - 2.0 * j_vec).unsqueeze(0).unsqueeze(1)
        cost = (weights * scores.unsqueeze(-1) - l1_sum.unsqueeze(-1)).permute(0, 2, 1)
        return torch.softmax(cost / tau, dim=-1)

    def diff_topk(self, scores, k=2, tau=0.2, num_bisection=15):
        lo = torch.min(scores, dim=-1, keepdim=True)[0] - 5.0 * tau
        hi = torch.max(scores, dim=-1, keepdim=True)[0] + 5.0 * tau
        for _ in range(num_bisection):
            mid = 0.5 * (lo + hi); w = torch.sigmoid((scores - mid) / tau)
            sum_w = torch.sum(w, dim=-1, keepdim=True)
            lo = torch.where(sum_w > float(k), mid, lo); hi = torch.where(sum_w > float(k), hi, mid)
        return torch.sigmoid((scores - 0.5 * (lo + hi)) / tau)

    def soft_histogram(self, x, bin_centers, bin_width=1.0, tau=0.1):
        lo = bin_centers.view(1, 1, -1) - 0.5 * bin_width; hi = bin_centers.view(1, 1, -1) + 0.5 * bin_width
        in_bin = torch.sigmoid((x.unsqueeze(-1) - lo) / tau) - torch.sigmoid((x.unsqueeze(-1) - hi) / tau)
        hist = torch.sum(in_bin, dim=1)
        return hist / (torch.sum(hist, dim=-1, keepdim=True) + 1e-12)

    def soft_quantile(self, scores, quantile=0.5, tau=0.2):
        B, N = scores.shape; target_rank = quantile * float(N - 1) + 1.0
        diff = (scores.unsqueeze(2) - scores.unsqueeze(1)) / tau
        ranks = 0.5 + torch.sum(torch.sigmoid(diff), dim=-1)
        weights = torch.softmax(-(ranks - target_rank).pow(2) / (2.0 * tau), dim=-1)
        return torch.sum(weights * scores, dim=-1)

model = DifferentiableSortingUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('DifferentiableSortingUnifiedModule initialized successfully.')

In [ ]:
p_soft, s_approx = model.soft_sort(weights['softsort_scores'], tau=0.2)
err_p_soft = (p_soft - weights['expected_p_soft']).abs().max().item()
err_s_approx = (s_approx - weights['expected_sorted_approx']).abs().max().item()
assert err_p_soft < 1e-5 and err_s_approx < 1e-5, f'SoftSort error: P={err_p_soft:.2e}, s={err_s_approx:.2e}'

p_neural = model.neural_sort(weights['neuralsort_scores'], tau=0.5)
err_p_neural = (p_neural - weights['expected_p_neural']).abs().max().item()
assert err_p_neural < 1e-5, f'NeuralSort error: P={err_p_neural:.2e}'

topk_w = model.diff_topk(weights['topk_scores'], k=2, tau=0.2, num_bisection=15)
err_topk = (topk_w - weights['expected_topk_weights']).abs().max().item()
assert err_topk < 1e-5, f'Diff Top-k error: {err_topk:.2e}'

hist_pred = model.soft_histogram(weights['hist_x'], weights['bin_centers'], bin_width=1.0, tau=0.1)
err_hist = (hist_pred - weights['expected_histogram']).abs().max().item()
assert err_hist < 1e-5, f'Soft histogram error: {err_hist:.2e}'

median_pred = model.soft_quantile(weights['quantile_scores'], quantile=0.5, tau=0.2)
err_med = (median_pred - weights['expected_median']).abs().max().item()
assert err_med < 1e-5, f'Soft quantile error: {err_med:.2e}'

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 6 invariants verified! SoftSort: {err_p_soft:.2e}/{err_s_approx:.2e}, NeuralSort: {err_p_neural:.2e}, TopK: {err_topk:.2e}, Hist: {err_hist:.2e}, Quantile: {err_med:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'SoftSort (P & s)': lambda: model.soft_sort(weights['softsort_scores']),
    'NeuralSort (P)': lambda: model.neural_sort(weights['neuralsort_scores']),
    'Diff Top-k': lambda: model.diff_topk(weights['topk_scores']),
    'Soft Histogram': lambda: model.soft_histogram(weights['hist_x'], weights['bin_centers']),
    'Soft Quantile': lambda: model.soft_quantile(weights['quantile_scores']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Temperature Saturation & Gradient Vanishing in Continuous Sorting
# When temperature is set too small (tau = 0.001), softmax logits saturate to extreme values,
# causing derivative dP/ds -> 0 everywhere, extinguishing backpropagation signals!
s_sat = torch.tensor([[1.0, 2.0, 3.0, 4.0]], requires_grad=True, device=device)
_, s_sorted_sat = model.soft_sort(s_sat, tau=0.001)
grad_sat = torch.autograd.grad(s_sorted_sat.sum(), s_sat)[0]
print(f'[FAILURE DEMONSTRATED] Saturated SoftSort (tau=0.001) max grad variance: {grad_sat.var().item():.2e} (vanishing gradient!)')
assert grad_sat.var().item() < 1e-4, 'Expected saturated vanishing gradient dynamics'

# Under calibrated temperature (tau = 0.5), smooth ranking gradients flow robustly:
s_cal = torch.tensor([[1.0, 2.0, 3.0, 4.0]], requires_grad=True, device=device)
_, s_sorted_cal = model.soft_sort(s_cal, tau=0.5)
grad_cal = torch.autograd.grad(s_sorted_cal.sum(), s_cal)[0]
print(f'[STABILIZED] Calibrated SoftSort (tau=0.5) gradients: {grad_cal.tolist()} (smooth non-vanishing flow!)')
assert (grad_cal > 0.05).all().item(), 'Expected active non-vanishing gradient flow'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Differentiable sorting, ranking, top-k, and quantiles verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Continuous Permutations Enable End-to-End Ranking:** SoftSort and NeuralSort bypass the non-differentiable `argsort` bottleneck by replacing discrete permutations with row-stochastic matrices.
2. **Annealing Prevents Gradient Saturation:** Temperature $\tau$ must be annealed geometrically during training to avoid early gradient starvation while converging toward discrete permutations.
3. **Scalability via Permutahedron Projection:** For large sets ($N > 4096$), avoid $O(N^2)$ permutation matrices by using Permutahedron isotonic regression ($O(N \log N)$ time, $O(N)$ memory).

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 12_latent_decision_gates_highway_hard_concrete_entmax_and_ponder_halting](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/12_latent_decision_gates_highway_hard_concrete_entmax_and_ponder_halting.ipynb)